# Telegram to Google Drive Bot

A lightweight bot that receives files from you on Telegram and saves them to your Google Drive.

## How to use
1. Replace the credentials below with your own
2. Run all cells (Runtime → Run all)
3. Message your bot on Telegram to send files

> ⚠️ **Security note:** Your credentials are stored in this notebook. Delete or disconnect the notebook from Colab after you're done to protect your bot token.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install pyrogram tgcrypto

In [ ]:
import os
import time
import shutil
import asyncio
from datetime import datetime

from pyrogram import Client, filters, idle
from pyrogram.types import Message

# ============================================================
# CONFIGURATION
# Replace these with your own credentials.
# - BOT_TOKEN: Get from @BotFather on Telegram
# - API_ID & API_HASH: Get from https://my.telegram.org
# ============================================================
BOT_TOKEN = "YOUR_BOT_TOKEN_HERE"
API_ID = 0
API_HASH = "YOUR_API_HASH_HERE"

DRIVE_FOLDER = "/content/drive/MyDrive/TelegramUploads"
TEMP_FOLDER = "/content/tmp_telegram_uploads"

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(DRIVE_FOLDER, exist_ok=True)

app = Client(
    "telegram_drive_bot",
    api_id=API_ID,
    api_hash=API_HASH,
    bot_token=BOT_TOKEN,
)

# ============================================================
# HELPERS
# ============================================================

def safe_filename(name: str) -> str:
    bad_chars = ["/", "\\", ":", "*", "?", '"', "<", ">", "|"]
    for ch in bad_chars:
        name = name.replace(ch, "_")
    return name.strip()


def make_unique_name(original_name: str) -> str:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{ts}_{original_name}"


def get_message_file_name(message: Message) -> str:
    if message.document:
        return message.document.file_name or "file"
    if message.video:
        return "video.mp4"
    if message.audio:
        return message.audio.file_name or "audio.mp3"
    if message.voice:
        return "voice.ogg"
    if message.photo:
        return "photo.jpg"
    if message.animation:
        return "animation.mp4"
    return "file"


# ============================================================
# HANDLERS
# ============================================================

@app.on_message(filters.private & (
    filters.document | filters.video | filters.audio |
    filters.voice | filters.photo | filters.animation
))
async def handle_file(client, message: Message):
    try:
        user = message.from_user
        user_id = user.id if user else "unknown"

        original_name = get_message_file_name(message)
        original_name = safe_filename(original_name)
        final_name = make_unique_name(original_name)

        temp_path = os.path.join(TEMP_FOLDER, final_name)
        drive_path = os.path.join(DRIVE_FOLDER, final_name)

        await message.reply_text("Processing your file... ⏳")

        await message.download(file_name=temp_path)
        shutil.move(temp_path, drive_path)

        text = (
            f"✅ File saved successfully.\n\n"
            f"📄 Name: `{final_name}`\n"
            f"📁 Path in Drive:\n`{drive_path}`\n"
            f"👤 User ID: `{user_id}`"
        )
        await message.reply_text(text, parse_mode="markdown")

    except Exception as e:
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Error: {e}")
        await message.reply_text(f"❌ Error processing file:\n`{e}`", parse_mode="markdown")


@app.on_message(filters.private & filters.command(["start", "help"]))
async def start_handler(client, message: Message):
    await message.reply_text(
        "👋 Hello! I'm your Telegram-to-Google Drive helper.\n\n"
        "Simply send me any file (document, video, photo, audio, voice)\n"
        "and I'll save it to your Google Drive.\n\n"
        "Supported types: Documents, Videos, Audio, Photos, Voice, GIFs."
    )


# ============================================================
# RUN BOT
# ============================================================

print("Starting bot...")
asyncio.create_task(run_bot := app.start())

## Keep the bot running

Run this cell to keep the bot alive. Leave Colab open and the kernel running.

> To stop: interrupt the runtime (Runtime → Interrupt).

In [ ]:
print("Bot is running. Message your bot on Telegram to send files.")
await app.start()
print("Bot is running...")
await idle()